# 自定义层

有时候，我们需要自己发明一个现在深度学习框架中还不存在的层。

## 不带参数的层

首先定义一个没有任何参数的自定义层（减去均值）， 构建他只需要继承基础层类并实现前向传播功能即可

In [1]:
import torch
import torch.nn.functional as F
from torch import nn


class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


向该层提供一些数据，验证他是否能按照预期进行工作

In [2]:
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

现在把这个层用在更复杂的模型中

In [3]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())

In [5]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(2.7940e-09, grad_fn=<MeanBackward0>)

## 带参数的层

这里定义有参数的层，这些参数可以通过训练进行调整。使用内置函数来创建参数。

在这里我们不需要为每个自定义层编写自定义的序列化程序。

现在我们实现自定义版本的全连接层，需要实现两个参数，一个表示权重，一个表示偏置项。

In [ ]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        # nn.Parameter 是一个特殊的Tensor包装器，这个张量是网络中的可训练参数，需要在反向传播的时候自动计算梯度并更新
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

实例化Mylinear类并访问其模型参数

In [7]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[ 0.8456, -0.4984, -0.9899],
        [ 0.0844, -0.9855, -0.3716],
        [ 1.4102, -1.5690,  0.5891],
        [-1.1125,  1.6265,  0.5849],
        [-0.4561, -0.5892,  0.3887]], requires_grad=True)

使用自定义层执行前向传播计算

In [8]:
linear(torch.rand(2, 5))

tensor([[0.3285, 0.9290, 0.1365],
        [0.5180, 0.0774, 0.2851]])

也可以使用自定义层去构建模型，就像使用内置的全连接层一样使用自定义层

In [9]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[13.5919],
        [11.7718]])

## Practice
* 设计一个可以接受输入并计算张量降维的层，返回$y_k = \sum_{i, j}W_{ijk}x_{i}x_{j}$

In [11]:
import torch
from torch import nn

class ReductionLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # 初始化权重 W，形状为 (out_features, in_features, in_features)
        # 即对应公式中的 W_{ijk}，其中 k 是 out_features, i 和 j 是 in_features
        self.W = nn.Parameter(torch.randn(out_features, in_features, in_features))
        
    def forward(self, x):
        # 假设输入 x 的形状是 (batch_size, in_features)
        # 我们需要计算 y_k = \sum_{i,j} W_{kij} * x_i * x_j
        # 使用 einsum: 
        # 'bi' 是批量输入 x 的形状
        # 'bj' 是同样批量输入 x 的形状 (同一份数据)
        # 'kij' 是权重 W 的形状
        # 'bk' 是输出结果的形状
        return torch.einsum('bi, bj, kij -> bk', x, x, self.W)

        # torch.einsum('输入张量的维度字母 -> 输出张量的维度字母', 张量1, 张量2, ...)
        #匹配即相乘：如果同一个字母在不同的输入张量中出现了，说明要在这些维度上做元素相乘（或者叫对齐）。
        #消失即求和：如果一个字母在箭头左边（输入）出现了，但在箭头右边（输出）没有出现，说明要在该维度上把数据全部加起来（求和/降维）。

# 测试该层
in_dim, out_dim = 4, 2
layer = ReductionLayer(in_dim, out_dim)
x = torch.rand(5, in_dim)  # 5 个样本，每个样本维度为 4
out = layer(x)
print(out)  # 期望输出形状: [5, 2]

tensor([[-2.7377, -1.8682],
        [-0.1769, -0.5699],
        [-1.7708, -1.9157],
        [ 0.2771,  0.5338],
        [ 0.0404,  0.1139]], grad_fn=<ViewBackward0>)


* 返回输入数据的傅里叶系数前半部分的层

假设我们有一个长度为 $N$ 的一维输入数据序列 $x = [x_0, x_1, ..., x_{N-1}]$。它的第 $k$ 个傅里叶系数 $X_k$ 的计算公式如下：$$X_k = \sum_{n=0}^{N-1} x_n \cdot e^{-j \frac{2\pi}{N} k n}$$

1. $x_n$：是你的输入数据（比如时域上的采样点）；
2. $e^{-j \frac{2\pi}{N} k n}$：根据欧拉公式（$e^{-j\theta} = \cos\theta - j\sin\theta$），它本质上代表了一个特定频率 $k$ 的正弦波/余弦波探测器；
3. 乘积求和（$\sum$）：这是在计算输入数据 $x_n$ 与特定频率探测波的相似度（内积）。如果输入数据中含有很强的频率 $k$ 的成分，乘积累加后的结果就会很大；如果不含有，累加后就会正负抵消趋近于 0。

In [13]:
import torch
from torch import nn

class HalfFFTLayer(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        # 对输入在最后一个维度上进行 1D 离散傅立叶变换
        # 注意: 较新的 PyTorch 版本推荐使用 torch.fft 模块
        fft_result = torch.fft.fft(x)
        
        # 获取特征维度的长度
        seq_len = x.shape[-1]
        
        # 返回傅立叶变换系数的前半部分
        # //2 确保在奇数长度时也是向下取整
        return fft_result[..., :seq_len // 2] # 前面的维度全部保留，最后一个维度只要前半部分

# 测试该层
layer = HalfFFTLayer()
x = torch.rand(2, 8)  # 批量大小 2，序列长度 8
out = layer(x)
print(out)  # 期望输出形状: [2, 4]，因为原始长度是8，一半是4

tensor([[ 4.4234+0.0000j,  1.3131+0.1060j,  0.4361-0.7044j, -0.1336-0.0926j],
        [ 5.2792+0.0000j, -0.6572+0.8411j, -0.0151+0.8882j, -0.0368+0.4898j]])
